# Supervised Model
- Collect images from dataset and transform into pixel array to train AI

### Load images from dataset in a vector

In [1]:
import os
import importlib
import CvrieLib.preprocessing as prep

importlib.reload(prep)
preprocess_image = prep.preprocess_image
flatten_image_array = prep.flatten_image_array

import numpy as np

X = []
Y = []

img_extension = '.jpg'

def load_fracture_dataset(directory, label_value, target_size=(224, 224)):
    data = []
    labels = []
    files = os.listdir(directory)
    for filename in files:
        if filename.lower().endswith(img_extension):
            full_path = os.path.join(directory, filename)
            try:
                img_vector = preprocess_image(full_path, size=target_size)
                data.append(img_vector)
                labels.append(label_value)
            except Exception as e:
                print(f"Erreur sur le fichier {filename}: {e}", file=os.sys.stderr)
    return data, labels


### Put label on data, and create X (the dataset ) and y the array of labels to keep track in next steps

In [2]:
X_fractured, y_fractured = load_fracture_dataset('Dataset/fractured/', label_value=1)
X_not_fractured, y_not_fractured = load_fracture_dataset('Dataset/not_fractured/', label_value=0)
# crer une pipeline avec scikit learn

X = np.concatenate([X_fractured, X_not_fractured], axis=0)
y = np.concatenate([y_fractured, y_not_fractured], axis=0)

print(f"Structure de X avant flatten : {X.shape}")

Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 224
(224, 224)
Resizing image to:  224 2

### Flatten the matrix in the matrix_array to obtain an array of array to comply model standard

In [3]:
X_flat = flatten_image_array(X)

print(f"Structure de X après flatten : {X_flat.shape}")

Structure de X après flatten : (9463, 50176)


#### Import scikit_learn

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

### Split dataset in 80/20, 80% to train and 20% to test, shuffled equally with stratify

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=5, stratify=y)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (7570, 50176), y_train shape: (7570,)


### GridSearch on the dataset to find the best model to use

In [6]:
param_grid_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=0), param_grid_rf, cv=5, scoring='f1', n_jobs=4)

grid_rf.fit(X_train, y_train)

print(f"Best RF Params: {grid_rf.best_params_}")
print(f"Best RF F1-Score: {grid_rf.best_score_}")

best_model = grid_rf.best_estimator_

KeyboardInterrupt: 

In [ ]:
y_prediction = best_model.predict(X_test)
print(confusion_matrix(y_test, y_prediction))
print(classification_report(y_test, y_prediction))

### Train on best Model and gives scores

In [ ]:
model = RandomForestClassifier(random_state=0, max_depth=2)
# faire du grid search sur les models

# puis faire du gradient free routines (soit celui de scikit learn soit notre propre)
# pour choisir les hyperparameters du modele

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("Accuracy Score:", accuracy)
print(f"Fractured samples: {len(y_fractured)}")
print(f"Not fractured samples: {len(y_not_fractured)}")

# tester le model sur une image spécifique